In [1]:
!which pip
!which python
import sys
print(sys.executable)

/home/zzou/.dataset/bin/pip
/home/zzou/.dataset/bin/python
/home/zzou/.dataset/bin/python


In [1]:
!export CUDA_VISIBLE_DEVICES=1
!nvidia-smi


Wed Feb 11 19:06:59 2026       
+---------------------------------------------------------------------------------------+
| NVIDIA-SMI 535.288.01             Driver Version: 535.288.01   CUDA Version: 12.2     |
|-----------------------------------------+----------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id        Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |         Memory-Usage | GPU-Util  Compute M. |
|                                         |                      |               MIG M. |
|=========================================+======================+======================|
|   0  NVIDIA RTX A4500               Off | 00000000:17:00.0 Off |                  Off |
| 30%   33C    P8              18W / 200W |     11MiB / 20470MiB |      0%      Default |
|                                         |                      |                  N/A |
+-----------------------------------------+----------------------+--

In [ ]:
from pathlib import Path
import os, glob

INPUT_DIR = Path("/home/zzou/WhatsUp_dataset/controlled_images")
OUTPUT_DIR = Path("/home/zzou/Version0_dataset/kontext_v0")
META_CSV  = OUTPUT_DIR / "meta_v0.csv"

os.makedirs(OUTPUT_DIR, exist_ok=True)

# image search
image_paths = sorted(sum([
    glob.glob(f"{INPUT_DIR}/**/*.{ext}", recursive=True)
    for ext in ["jpg","jpeg","png","webp"]
], []))

print(f"Found in total {len(image_paths)} images under {INPUT_DIR}")
assert len(image_paths) > 0, "!!! No image detected !!!"

Found in total 421 images under /home/zzou/WhatsUp_dataset/controlled_images


### A. Filter samples & Build metadata
For now, only use images containing "left & right" spatial relations.
Only use images with chair as object A.

For each selected initial image, use 3 different prompts to modify it (human face camera / face left / face right) and generate 3 edited image. Save necessary information in metadata

In [4]:
# filter base images
def match_name(fname: str) -> bool:
    fname = fname.lower()
    return ("_chair" in fname) and ("left" in fname or "right" in fname)

filtered_paths = [p for p in image_paths if match_name(os.path.basename(p))]
print(f"After filtering, will operate on {len(filtered_paths)} images.")

After filtering, will operate on 90 images.


In [ ]:
# Flux + GGUF inference example
import torch

from diffusers import FluxPipeline, FluxTransformer2DModel, GGUFQuantizationConfig

ckpt_path = (
    # "https://huggingface.co/QuantStack/Qwen-Image-Edit-2509-GGUF/resolve/main/Qwen-Image-Edit-2509-8B.Q8_0.gguf"
    "/home/zzou/ComfyUI/models/unet/Qwen-Image-Edit-2509-Q8_0.gguf"
)
transformer = FluxTransformer2DModel.from_single_file(
    ckpt_path,
    quantization_config=GGUFQuantizationConfig(compute_dtype=torch.bfloat16),
    torch_dtype=torch.bfloat16,
)
pipe = FluxPipeline.from_pretrained(
    "black-forest-labs/FLUX.1-dev",
    transformer=transformer,
    torch_dtype=torch.bfloat16,
)
pipe.enable_model_cpu_offload()
prompt = "A cat holding a sign that says hello world"
image = pipe(prompt, generator=torch.manual_seed(0)).images[0]
image.save("flux-gguf.png")

In [6]:
# Qwen-Image + GGUF
import os
import torch
from PIL import Image

from diffusers import QwenImageTransformer2DModel,QwenImageEditPlusPipeline,GGUFQuantizationConfig
from diffusers.utils import load_image

model_path = (
    # "https://huggingface.co/QuantStack/Qwen-Image-Edit-2509-GGUF/blob/main/"
    # "Qwen-Image-Edit-2509-Q3_K_S.gguf"
    "https://huggingface.co/QuantStack/Qwen-Image-Edit-2509-GGUF/blob/main/Qwen-Image-Edit-2509-Q8_0.gguf"
    # "/home/zzou/ComfyUI/models/unet/Qwen-Image-Edit-2509-Q8_0.gguf"
)

transformer = QwenImageTransformer2DModel.from_single_file(
    model_path,
    quantization_config=GGUFQuantizationConfig(compute_dtype=torch.bfloat16),
    torch_dtype=torch.bfloat16,
    config="callgg/image-edit-plus",
    subfolder="transformer",
)

pipe = QwenImageEditPlusPipeline.from_pretrained(
    "Qwen/Qwen-Image-Edit-2509",
    transformer=transformer,
    torch_dtype=torch.bfloat16,
)

print("pipeline loaded")
pipe.enable_model_cpu_offload()  # 显存不够建议开这个
# pipe.to("cuda:0") # Q3_0 oom
# pipe.transformer.to("cuda:0")
# pipe.text_encoder.to("cpu")
# pipe.vae.to("cpu")


Loading pipeline components...:   0%|          | 0/6 [00:00<?, ?it/s]

`torch_dtype` is deprecated! Use `dtype` instead!


Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

pipeline loaded


In [ ]:

def pick_dtype():
    # Ada/L4 一般支持 bf16，但为稳妥可用 fp16
    try:
        torch.zeros(1, dtype=torch.bfloat16, device="cuda")
        print("bf16 used!")
        return torch.bfloat16
    except Exception:
        return torch.float16



In [7]:
# EDIT_PROMPT_FACE_CAMERA_BASE  = ("Add one realistic human sitting on the existing chair, facing camera. Do not add new chairs. The chair may rotate. The human must face the camera directly. Do not block the f{second_object}. Match the image style and avoid artifacts.")

# EDIT_PROMPT_FACE_LEFT_BASE  = ("Add one realistic human sitting on the existing chair, facing left. Do not add new chairs. The chair may rotate. The human must face exactly 90 degree left in clean profile view. Do not block the f{second_object}. Match the image style and avoid artifacts.")

# EDIT_PROMPT_FACE_RIGHT_BASE  = ("Add one realistic human sitting on the existing chair, facing right. Do not add new chairs. The chair may rotate. The human must face exactly 90 degree right in clean profile view. Do not block the f{second_object}. Match the image style and avoid artifacts.")

EDIT_PROMPT_FACE_CAMERA_BASE  = ("Insert a human sitting on the chair facing the camera, with their body also faced the camera. Keep all the rest of the image the same.")
EDIT_PROMPT_FACE_LEFT_BASE  = ("Insert a human sitting on the chair facing left, with their body also turned to the left. Keep all the rest of the image the same.")
EDIT_PROMPT_FACE_RIGHT_BASE  = ("Insert a human sitting on the chair facing right, with their body also turned to the right. Keep all the rest of the image the same.")

In [8]:
OUTPUT_DIR = Path("/home/zzou/Version0_dataset/QwenImage_v0")
os.makedirs(OUTPUT_DIR, exist_ok=True)

In [9]:
# def resize_longest_side(img: Image.Image, max_side=1536):
#     w,h = img.size
#     m = max(w,h)
#     if m <= max_side:
#         return img
#     scale = max_side / m
#     return img.resize((int(w*scale), int(h*scale)), Image.LANCZOS)

# ================================================
import csv
from tqdm import tqdm

with open(META_CSV, "w", newline="", encoding="utf-8") as f:
    writer = csv.writer(f)
    writer.writerow(["src_path", "dst_path", "prompt", "expected_direction", "second_object", "view_type"])

    for src in tqdm(filtered_paths):
        img = load_image(src).convert("RGB")
        w,h = img.size

        second_object = os.path.basename(src).split("_", 1)[0]   # For example, src = /content/data/controlled_images/ball-of-yarn_left_of_chair.jpeg
        # print(second_object)
        EDIT_PROMPT_FACE_CAMERA = EDIT_PROMPT_FACE_CAMERA_BASE.format(second_object=second_object)
        EDIT_PROMPT_FACE_LEFT = EDIT_PROMPT_FACE_LEFT_BASE.format(second_object=second_object)
        EDIT_PROMPT_FACE_RIGHT = EDIT_PROMPT_FACE_RIGHT_BASE.format(second_object=second_object)

        # save image
        rel = os.path.relpath(src, INPUT_DIR)
        dst_dir = os.path.join(OUTPUT_DIR, os.path.dirname(rel))
        os.makedirs(dst_dir, exist_ok=True)
        base = os.path.splitext(os.path.basename(src))[0]

        
        ### inference 3 times for each image - 1
        # for EDIT_PROMPT in [EDIT_PROMPT_FACE_CAMERA, EDIT_PROMPT_FACE_LEFT, EDIT_PROMPT_FACE_RIGHT]:
        inputs = {
            "image": img,              # 对于 Plus 版本，也可以传 [img1, img2, ...]
            "prompt": EDIT_PROMPT_FACE_CAMERA,
            "generator": torch.manual_seed(0),
            "true_cfg_scale": 2.5,
            "negative_prompt": " ",
            "num_inference_steps": 20,
        }

        with torch.inference_mode():
            out = pipe(**inputs)
            out_img = out.images[0]
            out_path = os.path.join(dst_dir, f"{base}_FACE-CAMERA.png")
            out_img.save(out_path)
            # print("saved to:", os.path.abspath(out_path))

        # expected direction of the second object in view of the human figure
        if "left" in os.path.basename(src).lower():
            expected_direction_face_front = "right"
        elif "right" in os.path.basename(src).lower():
            expected_direction_face_front = "left"
        else:
            expected_direction_face_front = "unknown"
        view_type = "FACE-CAMERA"
        writer.writerow([src, out_path, EDIT_PROMPT_FACE_CAMERA, expected_direction_face_front , second_object, view_type])



        ### inference 3 times for each image -2
        inputs = {
            "image": img,              # 对于 Plus 版本，也可以传 [img1, img2, ...]
            "prompt": EDIT_PROMPT_FACE_LEFT,
            "generator": torch.manual_seed(0),
            "true_cfg_scale": 2.5,
            "negative_prompt": " ",
            "num_inference_steps": 20,
        }

        with torch.inference_mode():
            out = pipe(**inputs)
            out_img = out.images[0]
            out_path = os.path.join(dst_dir, f"{base}_FACE-LEFT.png")
            out_img.save(out_path)

        # expected direction of the second object in view of the human figure facing left
        if "left" in os.path.basename(src).lower():
            expected_direction_face_left = "front"
        elif "right" in os.path.basename(src).lower():
            expected_direction_face_left = "back"
        else:
            expected_direction_face_left = "unknown"
        view_type = "FACE-LEFT"
        writer.writerow([src, out_path, EDIT_PROMPT_FACE_LEFT, expected_direction_face_left , second_object, view_type])



        ### inference 3 times for each image -3
        inputs = {
            "image": img,              # 对于 Plus 版本，也可以传 [img1, img2, ...]
            "prompt": EDIT_PROMPT_FACE_RIGHT,
            "generator": torch.manual_seed(0),
            "true_cfg_scale": 2.5,
            "negative_prompt": " ",
            "num_inference_steps": 20,
        }

        with torch.inference_mode():
            out = pipe(**inputs)
            out_img = out.images[0]
            out_path = os.path.join(dst_dir, f"{base}_FACE-RIGHT.png")
            out_img.save(out_path)

        # expected direction of the second object in view of the human figure facing right
        if "left" in os.path.basename(src).lower():
            expected_direction_face_right = "back"
        elif "right" in os.path.basename(src).lower():
            expected_direction_face_right = "front"
        else:
            expected_direction_face_right = "unknown"
        view_type = "FACE-RIGHT"
        writer.writerow([src, out_path, EDIT_PROMPT_FACE_RIGHT, expected_direction_face_right , second_object, view_type])



print(f"Saved to {OUTPUT_DIR}. Initial images: {META_CSV}")


  0%|          | 0/90 [00:00<?, ?it/s]/home/zzou/.dataset/lib/python3.12/site-packages/torch/cuda/__init__.py:283: UserWarning: 
    Found GPU2 Quadro K620 which is of cuda capability 5.0.
    Minimum and Maximum cuda capability supported by this version of PyTorch is
    (7.0) - (12.0)
    
  warnings.warn(
/home/zzou/.dataset/lib/python3.12/site-packages/torch/cuda/__init__.py:304: UserWarning: 
    Please install PyTorch with a following CUDA
    configurations:  12.6 following instructions at
    https://pytorch.org/get-started/locally/
    
  warnings.warn(matched_cuda_warn.format(matched_arches))
/home/zzou/.dataset/lib/python3.12/site-packages/torch/cuda/__init__.py:326: UserWarning: 
Quadro K620 with CUDA capability sm_50 is not compatible with the current PyTorch installation.
The current PyTorch install supports CUDA capabilities sm_70 sm_75 sm_80 sm_86 sm_90 sm_100 sm_120.
If you want to use the Quadro K620 GPU with PyTorch, please check the instructions at https://pytorch.o

  0%|          | 0/20 [00:00<?, ?it/s]

  0%|          | 0/20 [00:00<?, ?it/s]

  0%|          | 0/20 [00:00<?, ?it/s]

  1%|          | 1/90 [07:50<11:37:56, 470.52s/it]

...

  0%|          | 0/20 [00:00<?, ?it/s]

  0%|          | 0/20 [00:00<?, ?it/s]

  0%|          | 0/20 [00:00<?, ?it/s]

 11%|█         | 10/90 [1:22:06<10:56:53, 492.67s/it]


OSError: [Errno 28] No space left on device

In [5]:
import shutil
folder_path = "/content/data/controlled_images_kontext_v0"
shutil.make_archive("dataset_kontext_v0", "zip", folder_path)
from google.colab import files
files.download("dataset_kontext_v0.zip")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [7]:
!mkdir -p "/content/drive/MyDrive/PR_test_datasets"
!cp -r "/content/data/controlled_images_kontext_v0" "/content/drive/MyDrive/PR_test_datasets"

### B. Validate modified images using mLLMs (on TELECOM cluster)
For limited number of simple images, manual validation is easy.

### C. Build my VQA based on v0 dataset

In [ ]:
import csv
import json
import os

META_CSV = "/content/data/controlled_images_kontext_v0/meta_v0.csv"
VQA_JSONL = "/content/vqa_direction_dataset.jsonl"  # The name of output VQA jsonl file

OPTIONS = ["left", "right", "front", "back", "none of the above / unclear"]

def direction_to_label(direction: str):
    """
    Map expected_direction string to:
    - answer letter: 'A' ... 'E'
    - answer index: 0 ... 4
    """
    if not direction:
        direction = ""
    d = direction.strip().lower()

    if d == "left":
        return "A", 0
    elif d == "right":
        return "B", 1
    elif d == "front":
        return "C", 2
    elif d == "back":
        return "D", 3
    else:
        return "E", 4 # unknown, unclear, etc.

def build_vqa_from_meta(meta_csv_path, output_jsonl_path):
    num_rows = 0
    with open(meta_csv_path, newline="", encoding="utf-8") as f_in, \
         open(output_jsonl_path, "w", encoding="utf-8") as f_out:

        reader = csv.DictReader(f_in)

        for row in reader:
            src_path = row.get("src_path")
            dst_path = row.get("dst_path")
            expected_direction = row.get("expected_direction", "")
            second_object = row.get("second_object", "").strip()

            if not dst_path:
                continue

            # map to correct answer letter
            answer_letter, answer_index = direction_to_label(expected_direction)

            # if second_object is empty
            if not second_object:
                second_object = "the object"

            question = (
                "From the person's perspective in the image,"
                f"Where is the {second_object} relative to the person? "
                "Choose one option from A to E and answer only the capital letter."
            )

            example = {
                "image": dst_path,
                "question": question,
                "options": OPTIONS,
                "answer": answer_letter,
                "answer_index": answer_index,
                "meta": {
                    "src_path": src_path,
                    "prompt": row.get("prompt", ""),
                    "expected_direction": expected_direction,
                    "second_object": second_object,
                }
            }

            # write the example into jsonl
            f_out.write(json.dumps(example, ensure_ascii=False) + "\n")
            num_rows += 1

    print(f"VQA dataset saved to {output_jsonl_path}, total {num_rows} examples.")


build_vqa_from_meta(META_CSV, VQA_JSONL)


VQA dataset saved to /content/vqa_direction_dataset.jsonl, total 270 examples.
